# Inspect ChromaDB Chunks

This notebook loads your local ChromaDB collection and prints chunk text + metadata.


In [8]:
import chromadb
import pandas as pd
from pathlib import Path
import json

# Use the same Chroma persist dir as the backend.
PROJECT_ROOT = Path("/Users/ajeyds/Projects/Doc Gap Analysis")
PERSIST_DIR = str(PROJECT_ROOT / "data/chroma_db")
COLLECTION_NAME = "rag_chunks"

# Filter to a single uploaded file.
# Default: auto-pick the latest entry from data/kb_metadata.json.
SOURCE_PATH = None

try:
    meta_path = PROJECT_ROOT / "data/kb_metadata.json"
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    latest_entry = max(meta["files"].values(), key=lambda e: e.get("uploadedAt", ""))
    SOURCE_PATH = latest_entry.get("path")
except Exception as e:
    print("Could not auto-resolve SOURCE_PATH:", e)


In [13]:
# 1) Connect to ChromaDB
client = chromadb.PersistentClient(path=PERSIST_DIR)
collection = client.get_collection(name=COLLECTION_NAME)

print("Notebook cwd:", Path.cwd())
print("Using PERSIST_DIR:", PERSIST_DIR)
print("Collection:", COLLECTION_NAME)
print("Total items:", collection.count())


Notebook cwd: /Users/ajeyds/Projects/Doc Gap Analysis
Using PERSIST_DIR: /Users/ajeyds/Projects/Doc Gap Analysis/data/chroma_db
Collection: rag_chunks
Total items: 35


In [14]:
# 2) Retrieve data
# NOTE: We intentionally do NOT include embeddings to keep the output small.
include = ["documents", "metadatas"]

if SOURCE_PATH:
    data = collection.get(where={"source": SOURCE_PATH}, include=include)
else:
    # Pull only the first N for quick inspection.
    # If you want everything, increase or remove the limit.
    data = collection.get(include=include, limit=200)

ids = data.get("ids", [])
documents = data.get("documents", [])
metadatas = data.get("metadatas", [])

print("Fetched rows:", len(ids))
print("Example metadata keys:", sorted((metadatas[0] or {}).keys()) if metadatas else [])


Fetched rows: 35
Example metadata keys: ['application', 'chunk_id', 'chunk_type', 'document_summary', 'document_title', 'document_type', 'epic', 'parent_context', 'source', 'total_acceptance_criteria', 'total_stories']


In [15]:
# 3) Format into a table
rows = []
for _id, doc, meta in zip(ids, documents, metadatas):
    meta = meta or {}
    rows.append(
        {
            "id": _id,
            "chunk_type": meta.get("chunk_type"),
            "chunk_id": meta.get("chunk_id"),
            "parent_context": meta.get("parent_context"),
            "source": meta.get("source"),
            "document": doc,
        }
    )

df = pd.DataFrame(rows, columns=["id", "chunk_type", "chunk_id", "parent_context", "source", "document"])
df = df.sort_values(by=["chunk_type", "chunk_id"], na_position="last")

# Show a preview (no embeddings)
df.head(30)


,id,chunk_type,chunk_id,parent_context,source,document
1,7942fea383c50d809eb43c340a3b0a8f55451d54db7f06...,ac,AC-1.1,Vendor Master Synchronization,/Users/ajeyds/Projects/Doc Gap Analysis/data/u...,SAP Sync Execution — Given the SAP RFC interfa...
2,3b991ae600985009e338917132e5d4d097bdcb65e4f024...,ac,AC-1.2,Vendor Master Synchronization,/Users/ajeyds/Projects/Doc Gap Analysis/data/u...,Mandatory Field Validation — Given vendor data...
3,438c110ac1a871268bc7c0d790dff97eed359bcf1076a4...,ac,AC-1.3,Vendor Master Synchronization,/Users/ajeyds/Projects/Doc Gap Analysis/data/u...,Unique Constraint — Given Vendor Code + Compan...
4,36c9e955e46b73a60208a8a44688a1f6fa44c5ba6878c4...,ac,AC-1.4,First-Time Vendor Qualification Setup,/Users/ajeyds/Projects/Doc Gap Analysis/data/u...,Record Creation — Given a vendor is synced fro...
5,f8af89c241ab357da32522941e5a1cd8b42f247fcd621b...,ac,AC-1.5,First-Time Vendor Qualification Setup,/Users/ajeyds/Projects/Doc Gap Analysis/data/u...,Duplicate Prevention — Given qualification rec...
6,8d6602a08ef07f3bc743a47fc84c80fec1f0bfa1d80668...,ac,AC-1.6,First-Time Vendor Qualification Setup,/Users/ajeyds/Projects/Doc Gap Analysis/data/u...,Audit Trail — Given any create/update/delete a...
7,83c0ac92a2ab5fb0887aa5688a9f913d36d0212c7eed86...,ac,AC-1.7,Vendor Eligibility Control,/Users/ajeyds/Projects/Doc Gap Analysis/data/u...,Inactive Vendor Selection — Inactive vendors c...
8,ffe1f1f48cdd5449322c95094a67b0fcdc9aacd9065824...,ac,AC-1.8,Vendor Eligibility Control,/Users/ajeyds/Projects/Doc Gap Analysis/data/u...,Eligibility Flag — Eligibility flag must be ro...
9,3efcc168bebea8fef7947231f227e721b2e13dd20ba1b8...,ac,AC-1.9,Vendor Eligibility Control,/Users/ajeyds/Projects/Doc Gap Analysis/data/u...,Audit Logging — Changes must be audit logged.
10,e480311781bb64de0df8097fdcdc481a1ab9925f3bfa23...,ac,AC-2.1,Automated Requalification Scheduling,/Users/ajeyds/Projects/Doc Gap Analysis/data/u...,Initiation Date Calculation — Initiation Date ...


In [18]:
print(df.loc[1, 'document'])

SAP Sync Execution — Given the SAP RFC interface is active, when the hourly job runs, then vendor data shall be fetched and updated in InLumin, duplicates shall be prevented, and sync logs shall be stored.


In [16]:
# 4) Print chunk text (optionally truncate)
N = 10
max_chars = 800  # set to None for full text

for i, row in df.head(N).iterrows():
    print("\n" + "=" * 80)
    print(f"[{row['chunk_type']}] chunk_id={row['chunk_id']}")
    print(f"parent_context={row['parent_context']}")
    print(f"source={row['source']}")
    text = row["document"] or ""
    if max_chars is not None:
        text = text[:max_chars] + ("..." if len(text) > max_chars else "")
    print(text)



[ac] chunk_id=AC-1.1
parent_context=Vendor Master Synchronization
source=/Users/ajeyds/Projects/Doc Gap Analysis/data/uploads/kb/caa3bcee-0308-48a4-86dd-f3ff960b974f_Rise User Stories.docx
SAP Sync Execution — Given the SAP RFC interface is active, when the hourly job runs, then vendor data shall be fetched and updated in InLumin, duplicates shall be prevented, and sync logs shall be stored.

[ac] chunk_id=AC-1.2
parent_context=Vendor Master Synchronization
source=/Users/ajeyds/Projects/Doc Gap Analysis/data/uploads/kb/caa3bcee-0308-48a4-86dd-f3ff960b974f_Rise User Stories.docx
Mandatory Field Validation — Given vendor data is received, when mandatory fields are missing, then the record shall be rejected and logged with error status.

[ac] chunk_id=AC-1.3
parent_context=Vendor Master Synchronization
source=/Users/ajeyds/Projects/Doc Gap Analysis/data/uploads/kb/caa3bcee-0308-48a4-86dd-f3ff960b974f_Rise User Stories.docx
Unique Constraint — Given Vendor Code + Company combination exist